In [8]:
import numpy as np

rng = np.random.default_rng()

Субъект (особь) - набор индексов разработчиков, назначенных на каждую задачу по порядку. (по факту: особь - ответ)

In [9]:
CATEGORIES_COUNT = 4

In [844]:
with open("input.txt") as f:
    tasks_count = N = np.loadtxt(f, max_rows=1, dtype=np.int64)
    tasks_categories = np.loadtxt(f, max_rows=1, dtype=np.int64) - 1  # (n,)
    tasks_times = np.loadtxt(f, max_rows=1, dtype=np.float64)  # (n,)
    developers_count = M = np.loadtxt(f, max_rows=1, dtype=np.int64)
    developers_coefficients = np.loadtxt(f, max_rows=M, dtype=np.float64)  # (n, m)

In [761]:
def create_random_population(genes_count: int, subjects_count: int) -> np.ndarray:
    return rng.integers(0, developers_count, size=(subjects_count, genes_count))

In [793]:
I_categories = np.eye(CATEGORIES_COUNT, dtype=bool)

def get_fitness(population: np.ndarray) -> np.ndarray:
    assert population.ndim == 2

    times = developers_coefficients[population][..., I_categories[tasks_categories]]
    times *= tasks_times

    mask = population == np.arange(developers_count)[:, None, None]
    masked = np.where(mask, times, 0)
    ret = masked.sum(axis=-1).max(axis=0)
    return 1 / (1e-10 + ret)

In [564]:
def su_sampling(fitness: np.ndarray, n: int, start: float = None) -> np.ndarray:
    _fitness = fitness.copy()
    _fitness.sort()
    _fitness = _fitness[::-1]
    argsort = fitness.argsort()[::-1]

    fitness_cumsum = np.cumsum(_fitness)
    h = fitness_cumsum[-1] / n

    if start is None:
        start = rng.uniform(0, h)

    pointers = np.arange(start, start + (n - 0.5) * h, h)
    mask = pointers[:, None] < fitness_cumsum
    return argsort[mask.argmax(axis=-1)]

In [628]:
def one_point_crossover(population_l: np.ndarray, population_r: np.ndarray, point: np.ndarray) -> np.ndarray:
    mask = np.arange(population_l.shape[1]) < point[:, None]
    return np.where(mask, population_l, population_r)

In [776]:
def create_new_population(old_population: np.ndarray, sampled_idx: np.ndarray, count: int) -> np.ndarray:
    sampled_population = old_population[sampled_idx]

    subject1_idx = rng.choice(sampled_population.shape[0], count)
    subject2_idx = (subject1_idx + rng.integers(1, sampled_population.shape[0], count)) % sampled_population.shape[0]

    points = rng.choice(sampled_population.shape[1], count)

    new_population = one_point_crossover(sampled_population[subject1_idx], sampled_population[subject2_idx], points)

    return new_population

In [796]:
def create_mutations(population: np.ndarray, gen_mutation_chance: float = 0.01) -> np.ndarray:
    new_population = population.copy()

    mutated_mask = rng.random(new_population.shape) < gen_mutation_chance
    total_mutated = mutated_mask.sum()

    new_population[mutated_mask] += rng.integers(1, developers_count, total_mutated)
    new_population[mutated_mask] %= developers_count
    return new_population

# !

In [845]:
POPULATION_COUNT = 100

In [846]:
population = create_random_population(tasks_count, POPULATION_COUNT)
population

array([[3, 4, 9, ..., 6, 7, 2],
       [9, 3, 2, ..., 5, 3, 8],
       [0, 6, 8, ..., 2, 5, 4],
       ...,
       [2, 0, 1, ..., 1, 2, 9],
       [4, 6, 6, ..., 4, 1, 7],
       [3, 8, 4, ..., 5, 8, 4]], shape=(100, 1000))

In [885]:
ITERATIONS_NUMBER = 100

In [896]:
for i in range(ITERATIONS_NUMBER):
    fitness = get_fitness(population)
    sampled_subjects_idx = su_sampling(fitness, 10)
    new_population = create_new_population(population, sampled_subjects_idx, POPULATION_COUNT)
    population = create_mutations(new_population)

In [849]:
population

array([[2, 0, 8, ..., 0, 5, 5],
       [2, 0, 8, ..., 8, 5, 8],
       [2, 0, 8, ..., 0, 5, 5],
       ...,
       [2, 0, 8, ..., 0, 5, 5],
       [2, 0, 8, ..., 0, 5, 5],
       [2, 0, 8, ..., 8, 5, 8]], shape=(100, 1000))

In [874]:
saved_output_index = 0
with open(f"output_info.txt", "w") as fi:
    print("", end="", file=fi)

In [897]:
saved_output_index += 1

_m = get_fitness(population).max()
_am = get_fitness(population).argmax()
print(_m, _am)
with open(f"output_info.txt", "a") as fi:
    print(_m, _am, file=fi)

with open(f"output{saved_output_index}.txt", "w") as f:
    print(*population[get_fitness(population).argmax()], file=f)

0.0014129183121275844 70
